In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# read our results, the table with active targets and the DrugBank vocabulary
df = pd.read_excel("../data/drugs_for_each_case_in_one_sheet.xlsx")
df_active=pd.read_csv("../data/ph_active.csv")
df_vocab = pd.read_csv("../data/drugbank_vocabulary.csv")

In [3]:
# identify all the drugs and the target from our results
result = (
    df.rename(columns={"Control nodes": "Controls"})
    [["Drug", "Controls"]]
    .assign(Controls=lambda x: x["Controls"].str.split(r"\s*,\s*"))
    .explode("Controls")
    .groupby("Drug", as_index=False)["Controls"]
    .agg(lambda x: sorted(set(x)))
)

In [4]:
#add DrugBank ID
result["ID_DB"] = result["Drug"].map(
    df_vocab.set_index("Common name")["DrugBank ID"]
)

In [5]:
#filter the targets to see the pharmacologically active ones; then save
lookup = (
    df_active.assign(ids=df_active["Drug IDs"].str.split(r"\s*;\s*"))
       .set_index("Gene Name")["ids"]
       .to_dict()
)

def filter(row):
    id_b = str(row["ID_DB"])

    return [
        x
        for x in row["Controls"]
        if x in lookup and id_b in lookup[x]
    ]

result["Ph active"] = result.apply(filter, axis=1)


In [6]:
# make a dictionary with key = drug, values = targets
drug_table = pd.read_excel(Path("../data/drugs_with_targets.xlsx"))

drug_t = (
    drug_table
        .dropna(subset=[drug_table.columns[2], drug_table.columns[3]])
        .assign(Genes=lambda df: df.iloc[:, 3].str.split(";"))
        .explode("Genes")
        .assign(Genes=lambda df: df["Genes"].str.strip())
        .groupby(drug_table.columns[2])["Genes"]
        .unique()
        .apply(list)
        .to_dict()
)

result["No. of targets"] = result["Drug"].map(
    lambda x: len(drug_t.get(x, [""])[0].split(",")) if drug_t.get(x) else 0
)

In [7]:
# save to excel

with pd.ExcelWriter("../data/drug_on_off_target.xlsx", engine="xlsxwriter") as writer:
    result.to_excel(writer, sheet_name="Sheet1", index=False)

    workbook = writer.book
    worksheet = writer.sheets["Sheet1"]

    header_format = workbook.add_format({
        "bold": True
    })

    # Rescrie header-ul cu formatul dorit
    for col_num, value in enumerate(result.columns):
        worksheet.write(0, col_num, value, header_format)